In [ ]:
# Import libraries and configure paths
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 6)

DATA_PATH = "../data/raw/csic_2010.csv"
PLOTS_DIR = "../results/plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

In [ ]:
# Load the dataset and clean column names for analysis
df = pd.read_csv("../data/raw/csic_2010.csv")

if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

if "lenght" in df.columns and "length" not in df.columns:
    df = df.rename(columns={"lenght": "length"})

df.columns = [col.strip() for col in df.columns]
df["classification"] = pd.to_numeric(df["classification"], errors="coerce").astype("Int64")
df["class_name"] = df["classification"].map({0: "Normal", 1: "Attack"})

print("Dataset loaded from:", os.path.abspath(DATA_PATH))
print("Available columns:", df.columns.tolist())

In [ ]:
# Display basic dataset information
print("Shape:", df.shape)
print("\nData types:\n")
print(df.dtypes)
print("\nMissing values per column:\n")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

df.head()

In [ ]:
# Analyze class distribution and save the count plot
class_counts = df["classification"].value_counts().sort_index()
class_percentages = df["classification"].value_counts(normalize=True).sort_index() * 100

print("Classification counts:\n")
print(class_counts)
print("\nClassification percentages:\n")
print(class_percentages.round(2))

plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df, x="class_name", order=["Normal", "Attack"])
ax.set_title("Class Distribution of CSIC 2010 HTTP Traffic")
ax.set_xlabel("Classification")
ax.set_ylabel("Count")

for container in ax.containers:
    ax.bar_label(container, fmt="%d")

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "class_distribution.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Compare HTTP methods across normal and attack traffic
method_by_class = pd.crosstab(df["Method"], df["class_name"]).reindex(columns=["Normal", "Attack"], fill_value=0)
print(method_by_class)

method_plot_data = method_by_class.reset_index().melt(
    id_vars="Method",
    value_vars=["Normal", "Attack"],
    var_name="class_name",
    value_name="count"
)

plt.figure(figsize=(8, 5))
ax = sns.barplot(data=method_plot_data, x="Method", y="count", hue="class_name")
ax.set_title("HTTP Method Distribution by Classification")
ax.set_xlabel("HTTP Method")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "http_method_by_classification.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Measure URL length patterns by class
df["url_length"] = df["URL"].fillna("").str.len()
avg_url_length = df.groupby("class_name")["url_length"].mean().reindex(["Normal", "Attack"])

print("Average URL length by class:\n")
print(avg_url_length.round(2))

plt.figure(figsize=(10, 6))
sns.histplot(data=df, x="url_length", hue="class_name", bins=40, kde=True, stat="count", common_norm=False, alpha=0.45)
plt.title("URL Length Distribution by Classification")
plt.xlabel("URL Length")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "url_length_distribution.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Measure payload length patterns by class
df["content_length"] = df["content"].fillna("").str.len()
avg_content_length = df.groupby("class_name")["content_length"].mean().reindex(["Normal", "Attack"])

print("Average content length by class:\n")
print(avg_content_length.round(2))

plt.figure(figsize=(10, 6))
sns.histplot(data=df, x="content_length", hue="class_name", bins=40, kde=True, stat="count", common_norm=False, alpha=0.45)
plt.title("Content Length Distribution by Classification")
plt.xlabel("Content Length")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "content_length_distribution.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Count URL injection indicators across normal and attack traffic
special_patterns = {
    "quote": r"'",
    "sql_comment": r"--",
    "semicolon": r";",
    "equals": r"=",
    "less_than": r"<",
    "greater_than": r">",
    "UNION": r"(?i)UNION",
    "SELECT": r"(?i)SELECT",
    "script": r"(?i)script"
}

url_series = df["URL"].fillna("")
special_freq = pd.DataFrame(index=["Normal", "Attack"])

for feature_name, pattern in special_patterns.items():
    counts = url_series.str.count(pattern)
    feature_totals = counts.groupby(df["class_name"]).sum()
    special_freq[feature_name] = feature_totals.reindex(["Normal", "Attack"], fill_value=0)

special_freq = special_freq.fillna(0).astype(int)
print(special_freq)

special_plot_data = special_freq.reset_index().rename(columns={"index": "class_name"}).melt(
    id_vars="class_name",
    var_name="feature",
    value_name="frequency"
)

plt.figure(figsize=(14, 6))
ax = sns.barplot(data=special_plot_data, x="feature", y="frequency", hue="class_name")
ax.set_title("Frequency of Special Characters and Keywords in URLs")
ax.set_xlabel("Special Character / Keyword")
ax.set_ylabel("Frequency")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "url_special_character_frequency.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Print a compact EDA summary for the project report
total_samples = len(df)
attack_count = int((df["classification"] == 1).sum())
normal_count = int((df["classification"] == 0).sum())
most_common_attack_method = df.loc[df["classification"] == 1, "Method"].mode().iat[0]
avg_url_length_normal = df.loc[df["classification"] == 0, "url_length"].mean()
avg_url_length_attack = df.loc[df["classification"] == 1, "url_length"].mean()
url_length_difference = avg_url_length_attack - avg_url_length_normal

print(f"Total samples: {total_samples}")
print(f"Attack samples: {attack_count}")
print(f"Normal samples: {normal_count}")
print(f"Most common HTTP method in attacks: {most_common_attack_method}")
print(f"Average URL length (Normal): {avg_url_length_normal:.2f}")
print(f"Average URL length (Attack): {avg_url_length_attack:.2f}")
print(f"Average URL length difference (Attack - Normal): {url_length_difference:.2f}")